## 라쿠텐 단일 리뷰데이터 추출 코드

In [1]:
import requests
import json
import time
import pandas as pd

def get_rakuten_all_reviews(shop_id, item_id):
    all_reviews = []
    url = "https://web-gateway.rakuten.co.jp/review/itemshopreviewlist/get/v1"
    
    # ⚠️ 쿠키 인코딩 에러 방지 처리
    raw_cookie = r"_ra=1773145336745|9810936d-5cda-4630-8e49-c98379684815; Rp=dc4c4487b0d3c4dce21f7d6bdbf70329f03486c8; rcxGlobal=d45a9a4c-4afa-4759-a9ed-229577264fb8; bm_mi=02819FD1874A677A9FC3219D127ECD18~YAAQB9ojF3ykzaGcAQAAMAK01x85bJZA06vzzbG4DIKAXg7mopEgiKN5lzrV/0AErGfSM50oZ5QfIMqbrcROt7DYSkrYw58Cz7vwEO2XkjKMkHGsBRiolwhSHY6BjhqOg8UrHpjdOwbKwGHWbgXrhYVrXbpqnuYxOnS6Rd4tbRsuPIrmZrskES38smu59dSiBK0bGYfeffgKl/2sz7WT7sEbAOAom3GeHhE5vSnLp8Unxda00YTc54V1R0SccenoAvO/8tAjlqf4fcZMyVfgWjmkkXpBvobpj5hyrWagxWfFFgn06n/L/mwrI22pddxKke0Bln2zVz7w9QvyrImNfBfU7zu1YGnnS6HKKDs9fU21qukz3/qjiTlpM3yQIgMrYmSo2aL/dooM6uWJl/I7WeDfGzeHGvHpBy6zNQ==~1; bm_sv=F3073B8724961BB7DCBE5A9516653181~YAAQbIj+eWOAwsucAQAA7Di01x/kkLOsyzmShVkJPcJf1O0TkzKdl7ZL7qgaen5E33IuCROVCPlrVdQr767lxL25yZRnHKhQTLV74swQf/WGDo3WVE2ehk0W7uv6G6PUyNE2N5dQyBl/0tnQF1TIL1GoIY1QVyysRmPhNLu9lndOUVbr8CgHlwjK+8cZfXcWG/rrLNFiEsHCtxhiQUijgpAP4TGE1raLDQ8pAaWOcclxAHnAjyBcQ2TkH+CwcSLNeCnH~1; ak_bmsc=6DA2C442401E92959215D4FC32333198~000000000000000000000000000000~YAAQB9ojF5AVzqGcAQAA0cu01x9fj3L5e+d9RAJ1hoLlH3K8NWtbYVM99UU1cPhBjs0l+RT4L93zSS2KCBZ2cF79c9IkAVFdcgGHbxP50Uf8nlO5I6u+fsByVK6zS4EhUNSCSizuhMPBdf0lHio/+lSN+8w0fTRgZfWwPNbWNRlfyg8SAriAihvo7Qf2txj9kJZGnaHPqgkCZXs11UwhehZ5Ak3MzYtTcojSc91bzqp+zzmmEbyhuYzqW5FWGr6d5k6LF157rv/BKCBjyW2dtk7TbRgJKgQGZz/gPYD8n8mDMroN36ptPxuByW5GOOwTwoCs8tDnUOjgOtm8M7mmdRNKlpJsUSJoOylT7fkQoeVPZkHsLG1j5QOPG60+P7zXsIeB4pddGO/PKsDF8HPAgLpOVb2Dgqz56kyCR4vfPNXZLawqlDXl8+8HXh2AL5KJmQIZMy0XvT/jsyXGkSBk6KO8l/jsyQp1AG570i/fWulONUZGzkQKF86JqOOaoDS7dUrDgp1mqvDR4lwtHSs=; krt_rewrite_uid=f8b15f33-cc3e-45c0-b1be-9a29a4cff1d2; Re=31.1.5.0.0.216348.3-31.1.5.0.0.216348.3; rat_v=05252dcb01d4e6863512b1a3e769b00e711919a"
    safe_cookie = raw_cookie.encode('utf-8').decode('latin-1', 'ignore')

    headers = {
        "authkey": "isrlPcMjUuXCVBUTVh91ZcHEfoI45CmPR",
        "content-type": "application/json; charset=UTF-8",
        "accept": "application/json, text/plain, */*",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        "cookie": safe_cookie,
        "referer": "https://review.rakuten.co.jp/"
    }

    page = 1
    has_next = True

    print(f"🚀 상품 {item_id} 모든 리뷰 수집 시작 (전체 약 2,200여 건)...")

    while has_next:
        # ✅ 중첩 구조 Payload 반영
        payload = {
            "common": {
                "params": {"device": "pc"},
                "include": ["itemReviewList"]
            },
            "features": {
                "itemReviewList": {
                    "params": {
                        "shopId": int(shop_id),
                        "itemId": int(item_id),
                        "sort": "",
                        "page": str(page),
                        "hits": 30,
                        "filter": {"rating": "", "mediaOnly": "false", "ageRange": "", "sex": ""},
                        "includePickupReview": True
                    }
                }
            }
        }

        try:
            response = requests.post(url, headers=headers, json=payload)
            
            # 200 또는 207 코드가 오면 정상
            if response.status_code not in [200, 207]:
                print(f"\n❌ {page}p 중단 (코드:{response.status_code})")
                break

            data = response.json()
            
            # 파싱 경로: body -> itemReviewList -> data
            res_body = data.get("body", {}).get("itemReviewList", {})
            res_data = res_body.get("data", {})
            reviews = res_data.get("reviews", [])
            
            if not reviews:
                print(f"\n✅ {page}페이지 결과 없음. 수집 완료.")
                break
                
            for rev in reviews:
                all_reviews.append({
                    "Page": page,
                    "Nickname": rev.get("nickname"),
                    "Rating": rev.get("rating"),
                    "Body": rev.get("body"),
                    "PostDate": rev.get("postDate"),
                    "Age": f"{rev.get('ageRange', '')}{rev.get('ageSuffix', '')}",
                    "Sex": rev.get("sex"),
                    "Sku": rev.get("skuInfo")
                })
            
            # ✅ 다음 페이지 존재 여부 확인
            has_next = res_data.get("hasNextPage", False)
            
            print(f"🔄 {page}페이지 완료... (누적 {len(all_reviews)}개)", end='\r')
            
            page += 1
            # 라쿠텐 보안을 고려하여 1.5~2초 간격 권장
            time.sleep(1.8)

        except Exception as e:
            print(f"\n❌ 에러 발생: {e}")
            break

    # 최종 저장
    if all_reviews:
        filename = f"rakuten_{shop_id}_{item_id}_ALL.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(all_reviews, f, ensure_ascii=False, indent=4)
        print(f"\n📂 총 {len(all_reviews)}개 데이터 저장 완료! ('{filename}')")
    
    return pd.DataFrame(all_reviews)

# --- 실행 ---
SHOP_ID = "371043"
ITEM_ID = "10000027"
df = get_rakuten_all_reviews(SHOP_ID, ITEM_ID)

🚀 상품 10000027 모든 리뷰 수집 시작 (전체 약 2,200여 건)...
🔄 77페이지 완료... (누적 2293개)
📂 총 2293개 데이터 저장 완료! ('rakuten_371043_10000027_ALL.json')


## 라쿠텐 top10 추출코드

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time
import random
import json
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
TARGET_URL  = "https://ranking.rakuten.co.jp/weekly/100944/p=1/"
SAVE_FILE   = "rakuten_skincare_top10.json"

# ── Selenium 설정 ─────────────────────────────────────────────────
options = uc.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--incognito')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')

driver = None
try:
    driver = uc.Chrome(options=options)

    print("📡 라쿠텐 랭킹 페이지 접속 중...")
    driver.get(TARGET_URL)
    time.sleep(random.uniform(5, 8))

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # ── 상품 카드 수집 ─────────────────────────────────────────────
    # 1~3위: .rnkRanking_top3box / 4위~: .rnkRanking_after4box
    all_items = soup.select("div.rnkRanking_top3box, div.rnkRanking_after4box")
    print(f"📦 상품 카드 {len(all_items)}개 발견")

    final_rankings = []

    for item in all_items:
        if len(final_rankings) >= 10:
            break
        try:
            # A. 순위
            rank_icon = item.select_one(".rnkRanking_rankIcon img")        # 1~3위
            disp_rank = item.select_one(".rnkRanking_dispRank")            # 4위~
            if rank_icon:
                alt_text = rank_icon.get("alt", "")                        # "1位", "2位" ...
                rank_num = int(alt_text.replace("位", "")) if "位" in alt_text else len(final_rankings) + 1
            elif disp_rank:
                rank_text = disp_rank.get_text(strip=True).replace("位", "")
                rank_num = int(rank_text) if rank_text.isdigit() else len(final_rankings) + 1
            else:
                rank_num = len(final_rankings) + 1

            # B. 상품명 + URL
            title_tag = item.select_one(".rnkRanking_itemName a")
            if not title_tag:
                continue
            title = title_tag.get_text(strip=True)
            product_url = title_tag.get("href", "")

            # C. 숍명
            shop_tag = item.select_one(".rnkRanking_shop a")
            shop_name = shop_tag.get_text(strip=True) if shop_tag else "N/A"

            # D. 가격
            price_tag = item.select_one(".rnkRanking_price")
            price = price_tag.get_text(strip=True) if price_tag else "N/A"

            # E. 평점 (starON 개수 합산)
            stars_on   = len(item.select(".rnkRanking_starON"))
            stars_half = len(item.select(".rnkRanking_starHALF"))
            rating = stars_on + (0.5 if stars_half else 0)

            # F. 리뷰 수 + shop_id / item_id 추출
            # href 예: https://review.rakuten.co.jp/item/1/357686_10014729/1.1/
            review_link = item.select_one("a[href*='review.rakuten.co.jp/item']")
            reviews_cnt = 0
            shop_id, item_id = "", ""
            if review_link:
                rv_text = review_link.get_text(strip=True)  # "レビュー(1,778件)"
                reviews_cnt = int(
                    rv_text.replace("レビュー(", "").replace("件)", "").replace(",", "").strip()
                ) if "レビュー" in rv_text else 0

                href = review_link.get("href", "")
                parts = href.rstrip("/").split("/")
                id_part = next((p for p in reversed(parts) if "_" in p), "")
                if id_part:
                    shop_id, item_id = id_part.split("_", 1)

            # G. 순위 변동 (Up / Down / Stay)
            trend_img = item.select_one(".rnkRanking_preRank img")
            trend = "N/A"
            if trend_img:
                alt = trend_img.get("alt", "")
                trend = "Up" if "Up" in alt else ("Down" if "Down" in alt else "Stay")

            final_rankings.append({
                "rank"        : rank_num,
                "title"       : title,
                "shop_name"   : shop_name,
                "rating"      : rating,
                "reviews"     : reviews_cnt,
                "price"       : price,
                "trend"       : trend,
                "url"         : product_url,
                "shop_id"     : shop_id,
                "item_id"     : item_id,
                "platform"    : "Rakuten",
                "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })
            print(f"✅ {rank_num:>2}위 [{trend}] {shop_name[:15]} | {title[:30]} | ⭐{rating} ({reviews_cnt}건) | {price}")

        except Exception as e:
            print(f"⚠️ 파싱 오류: {e}")
            continue

    # ── 저장 ──────────────────────────────────────────────────────
    with open(SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump(final_rankings, f, ensure_ascii=False, indent=4)

    print()
    print("=" * 70)
    print("🏆 라쿠텐 스킨케어 랭킹 Top 10 (주간)")
    print("=" * 70)
    for item in final_rankings:
        print(f"{item['rank']:>2}위 [{item['trend']}] | {item['shop_name'][:15]:<15} | {item['title'][:25]:<25} | ⭐{item['rating']} | {item['price']}")
    print("=" * 70)
    print(f"\n💾 저장 완료: {SAVE_FILE} (총 {len(final_rankings)}개)")

except Exception as e:
    print(f"❌ 에러 발생: {e}")
finally:
    if driver:
        time.sleep(3)
        driver.quit()

📡 라쿠텐 랭킹 페이지 접속 중...
📦 상품 카드 80개 발견
✅  1위 [Up] VTcosmetic楽天市場店 | ＼最大49％OFF＋ギフト＋送料無料／総合ランキング1位【V | ⭐4.5 (7886건) | 2,783円
✅  2위 [Down] アテニア公式ショップ　楽天市場 | 【ポイント10倍！〜3月11日1:59】スキンクリア クレン | ⭐4.5 (44082건) | 3,630円～
✅  3위 [Up] VTcosmetic楽天市場店 | 【クーポン付き】＼最大68％OFF＋現品ギフト＋送料無料／【 | ⭐4.5 (1520건) | 4,900円～
✅  4위 [Up] 【公式】Yunth Store | 【P30%還元+セット9日23:59マデ】【公式】Yunth | ⭐4.5 (43354건) | 3,960円～
✅  5위 [Up] ANUA Official 楽 | ★58％OFF+現品ギフト+先着ギフトまで★【ROOMコラボ | ⭐3 (2건) | 6,800円
✅  6위 [Up] コスメティック　やよい | ＼超トクも残り2日!P20%還元確定+10%OFF／【資生堂 | ⭐4.5 (1778건) | 6,600円～
✅  7위 [Down] アテニア公式ショップ　楽天市場 | 【ポイント5倍！〜3月11日1:59】スキンクリア クレンズ | ⭐4.5 (37941건) | 1,980円
✅  8위 [Up] VTcosmetic楽天市場店 | ＼最大58％OFF＋ギフト＋送料無料／【 VT リードルショ | ⭐4.5 (506건) | 3,990円～
✅  9위 [Stay] シュウ ウエムラ 公式ショップ | 【ポイント10倍｜3/4 20:00-3/11 1:59】総 | ⭐4.5 (31569건) | 15,400円
✅ 10위 [Up] タカミ 公式ショップ楽天市場店 | 【ポイント10倍│楽天スーパーSALE限定販売】タカミ角質美 | ⭐4.5 (433건) | 11,440円

🏆 라쿠텐 스킨케어 랭킹 Top 10 (주간)
 1위 [Up] | VTcosmetic楽天市場店 | ＼最大49％OFF＋ギフト＋送料無料／総合ランキン | ⭐4.5 | 2,783円
 2위 [Down] | アテニア公式シ

## 라쿠텐 top10 추출 및 리뷰데이터 추출 코드

In [5]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import time
import random
import json
import pandas as pd
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
TARGET_URL       = "https://ranking.rakuten.co.jp/weekly/100944/p=1/"
TARGET_RANK      = 3      # ← 리뷰 수집할 순위
RANK_SAVE_FILE   = "rakuten_skincare_top10.jsonl"
REVIEW_SAVE_FILE = "rakuten_reviews_rank3.json"

# 라쿠텐 리뷰 API 헤더 (쿠키 만료 시 교체 필요)
RAW_COOKIE = r"_ra=1773145336745|9810936d-5cda-4630-8e49-c98379684815; Rp=dc4c4487b0d3c4dce21f7d6bdbf70329f03486c8; rcxGlobal=d45a9a4c-4afa-4759-a9ed-229577264fb8; bm_mi=02819FD1874A677A9FC3219D127ECD18~YAAQB9ojF3ykzaGcAQAAMAK01x85bJZA06vzzbG4DIKAXg7mopEgiKN5lzrV/0AErGfSM50oZ5QfIMqbrcROt7DYSkrYw58Cz7vwEO2XkjKMkHGsBRiolwhSHY6BjhqOg8UrHpjdOwbKwGHWbgXrhYVrXbpqnuYxOnS6Rd4tbRsuPIrmZrskES38smu59dSiBK0bGYfeffgKl/2sz7WT7sEbAOAom3GeHhE5vSnLp8Unxda00YTc54V1R0SccenoAvO/8tAjlqf4fcZMyVfgWjmkkXpBvobpj5hyrWagxWfFFgn06n/L/mwrI22pddxKke0Bln2zVz7w9QvyrImNfBfU7zu1YGnnS6HKKDs9fU21qukz3/qjiTlpM3yQIgMrYmSo2aL/dooM6uWJl/I7WeDfGzeHGvHpBy6zNQ==~1; bm_sv=F3073B8724961BB7DCBE5A9516653181~YAAQbIj+eWOAwsucAQAA7Di01x/kkLOsyzmShVkJPcJf1O0TkzKdl7ZL7qgaen5E33IuCROVCPlrVdQr767lxL25yZRnHKhQTLV74swQf/WGDo3WVE2ehk0W7uv6G6PUyNE2N5dQyBl/0tnQF1TIL1GoIY1QVyysRmPhNLu9lndOUVbr8CgHlwjK+8cZfXcWG/rrLNFiEsHCtxhiQUijgpAP4TGE1raLDQ8pAaWOcclxAHnAjyBcQ2TkH+CwcSLNeCnH~1; ak_bmsc=6DA2C442401E92959215D4FC32333198~000000000000000000000000000000~YAAQB9ojF5AVzqGcAQAA0cu01x9fj3L5e+d9RAJ1hoLlH3K8NWtbYVM99UU1cPhBjs0l+RT4L93zSS2KCBZ2cF79c9IkAVFdcgGHbxP50Uf8nlO5I6u+fsByVK6zS4EhUNSCSizuhMPBdf0lHio/+lSN+8w0fTRgZfWwPNbWNRlfyg8SAriAihvo7Qf2txj9kJZGnaHPqgkCZXs11UwhehZ5Ak3MzYtTcojSc91bzqp+zzmmEbyhuYzqW5FWGr6d5k6LF157rv/BKCBjyW2dtk7TbRgJKgQGZz/gPYD8n8mDMroN36ptPxuByW5GOOwTwoCs8tDnUOjgOtm8M7mmdRNKlpJsUSJoOylT7fkQoeVPZkHsLG1j5QOPG60+P7zXsIeB4pddGO/PKsDF8HPAgLpOVb2Dgqz56kyCR4vfPNXZLawqlDXl8+8HXh2AL5KJmQIZMy0XvT/jsyXGkSBk6KO8l/jsyQp1AG570i/fWulONUZGzkQKF86JqOOaoDS7dUrDgp1mqvDR4lwtHSs=; krt_rewrite_uid=f8b15f33-cc3e-45c0-b1be-9a29a4cff1d2; Re=31.1.5.0.0.216348.3-31.1.5.0.0.216348.3; rat_v=05252dcb01d4e6863512b1a3e769b00e711919a"
SAFE_COOKIE = RAW_COOKIE.encode('utf-8').decode('latin-1', 'ignore')

REVIEW_API_URL = "https://web-gateway.rakuten.co.jp/review/itemshopreviewlist/get/v1"
REVIEW_HEADERS = {
    "authkey"     : "isrlPcMjUuXCVBUTVh91ZcHEfoI45CmPR",
    "content-type": "application/json; charset=UTF-8",
    "accept"      : "application/json, text/plain, */*",
    "user-agent"  : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    "cookie"      : SAFE_COOKIE,
    "referer"     : "https://review.rakuten.co.jp/",
}


# ── 함수: 라쿠텐 리뷰 전체 수집 ──────────────────────────────────
def get_rakuten_all_reviews(shop_id, item_id, product_name):
    all_reviews = []
    page = 1
    has_next = True

    print(f"\n🚀 [{product_name}] 리뷰 전체 수집 시작... (shop={shop_id}, item={item_id})")

    while has_next:
        payload = {
            "common": {
                "params": {"device": "pc"},
                "include": ["itemReviewList"]
            },
            "features": {
                "itemReviewList": {
                    "params": {
                        "shopId"            : int(shop_id),
                        "itemId"            : int(item_id),
                        "sort"              : "",
                        "page"              : str(page),
                        "hits"              : 30,
                        "filter"            : {"rating": "", "mediaOnly": "false", "ageRange": "", "sex": ""},
                        "includePickupReview": True,
                    }
                }
            }
        }
        try:
            resp = requests.post(REVIEW_API_URL, headers=REVIEW_HEADERS, json=payload, timeout=15)
            if resp.status_code not in [200, 207]:
                print(f"\n❌ {page}p 중단 (코드: {resp.status_code})")
                break

            data     = resp.json()
            res_body = data.get("body", {}).get("itemReviewList", {})
            res_data = res_body.get("data", {})
            reviews  = res_data.get("reviews", [])

            if not reviews:
                print(f"\n✅ {page}페이지 결과 없음. 수집 완료.")
                break

            for rev in reviews:
                all_reviews.append({
                    "Page"    : page,
                    "Nickname": rev.get("nickname"),
                    "Rating"  : rev.get("rating"),
                    "Body"    : rev.get("body"),
                    "PostDate": rev.get("postDate"),
                    "Age"     : f"{rev.get('ageRange', '')}{rev.get('ageSuffix', '')}",
                    "Sex"     : rev.get("sex"),
                    "Sku"     : rev.get("skuInfo"),
                })

            has_next = res_data.get("hasNextPage", False)
            print(f"🔄 {page}페이지 완료... (누적 {len(all_reviews)}개)", end="\r")
            page += 1
            time.sleep(1.8)

        except Exception as e:
            print(f"\n❌ 에러: {e}")
            break

    print(f"\n✨ 리뷰 수집 완료: 총 {len(all_reviews)}개")
    return all_reviews


# ── STEP 1: Selenium으로 Top 10 랭킹 수집 ────────────────────────
options = uc.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--incognito')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')

driver = None
try:
    driver = uc.Chrome(options=options)

    print("📡 라쿠텐 랭킹 페이지 접속 중...")
    driver.get(TARGET_URL)
    time.sleep(random.uniform(5, 8))

    soup      = BeautifulSoup(driver.page_source, "html.parser")
    all_items = soup.select("div.rnkRanking_top3box, div.rnkRanking_after4box")
    print(f"📦 상품 카드 {len(all_items)}개 발견")

    rank_data_list = []
    review_master  = []
    rank_count     = 1

    for item in all_items:
        if rank_count > 10:
            break
        try:
            # 순위
            rank_icon = item.select_one(".rnkRanking_rankIcon img")
            disp_rank = item.select_one(".rnkRanking_dispRank")
            if rank_icon:
                alt = rank_icon.get("alt", "")
                rank_num = int(alt.replace("位", "")) if "位" in alt else rank_count
            elif disp_rank:
                t = disp_rank.get_text(strip=True).replace("位", "")
                rank_num = int(t) if t.isdigit() else rank_count
            else:
                rank_num = rank_count

            # 상품명 + URL
            title_tag = item.select_one(".rnkRanking_itemName a")
            if not title_tag:
                continue
            title       = title_tag.get_text(strip=True)
            product_url = title_tag.get("href", "")

            # 숍명
            shop_tag  = item.select_one(".rnkRanking_shop a")
            shop_name = shop_tag.get_text(strip=True) if shop_tag else "N/A"

            # 가격
            price_tag = item.select_one(".rnkRanking_price")
            price     = price_tag.get_text(strip=True) if price_tag else "N/A"

            # 평점
            stars_on   = len(item.select(".rnkRanking_starON"))
            stars_half = len(item.select(".rnkRanking_starHALF"))
            rating     = stars_on + (0.5 if stars_half else 0)

            # 리뷰 수 + shop_id / item_id
            review_link = item.select_one("a[href*='review.rakuten.co.jp/item']")
            reviews_cnt = 0
            shop_id = item_id = ""
            if review_link:
                rv_text     = review_link.get_text(strip=True)
                reviews_cnt = int(
                    rv_text.replace("レビュー(", "").replace("件)", "").replace(",", "").strip()
                ) if "レビュー" in rv_text else 0
                href    = review_link.get("href", "")
                parts   = href.rstrip("/").split("/")
                id_part = next((p for p in reversed(parts) if "_" in p), "")
                if id_part:
                    shop_id, item_id = id_part.split("_", 1)

            # 순위 변동
            trend_img = item.select_one(".rnkRanking_preRank img")
            trend = "N/A"
            if trend_img:
                alt   = trend_img.get("alt", "")
                trend = "Up" if "Up" in alt else ("Down" if "Down" in alt else "Stay")

            rank_data_list.append({
                "rank"        : rank_num,
                "title"       : title,
                "shop_name"   : shop_name,
                "rating"      : rating,
                "reviews"     : reviews_cnt,
                "price"       : price,
                "trend"       : trend,
                "url"         : product_url,
                "shop_id"     : shop_id,
                "item_id"     : item_id,
                "platform"    : "Rakuten",
                "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })
            print(f"📍 {rank_num:>2}위 [{trend}] {shop_name[:15]} | {title[:30]} | ⭐{rating} ({reviews_cnt}건)")

            # ── STEP 2: 3위 상품만 리뷰 수집 ─────────────────────
            if rank_num == TARGET_RANK and shop_id and item_id:
                review_master = get_rakuten_all_reviews(shop_id, item_id, title)

            rank_count += 1

        except Exception as e:
            print(f"⚠️ {rank_count}위 파싱 오류: {e}")
            rank_count += 1
            continue

    # ── STEP 3: 저장 ──────────────────────────────────────────────
    with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
        for entry in rank_data_list:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump(review_master, f, ensure_ascii=False, indent=4)

    # ── STEP 4: 결과 출력 ─────────────────────────────────────────
    print()
    print("=" * 70)
    print("🏆 라쿠텐 스킨케어 랭킹 Top 10 (주간)")
    print("=" * 70)
    for item in rank_data_list:
        marker = " ◀ 리뷰 수집됨" if item["rank"] == TARGET_RANK else ""
        print(f"{item['rank']:>2}위 [{item['trend']}] | {item['shop_name'][:15]:<15} | {item['title'][:25]:<25} | ⭐{item['rating']} | {item['price']}{marker}")
    print("=" * 70)
    print(f"\n📂 저장 완료")
    print(f"  - 순위 파일 : {RANK_SAVE_FILE}")
    print(f"  - 리뷰 파일 : {REVIEW_SAVE_FILE} (총 {len(review_master)}개)")

    if review_master:
        df = pd.DataFrame(review_master)
        print(f"\n📋 3위 상품 리뷰 상위 5개")
        display(df.head())

except Exception as e:
    print(f"❌ 에러 발생: {e}")
finally:
    if driver:
        time.sleep(3)
        driver.quit()

📡 라쿠텐 랭킹 페이지 접속 중...
📦 상품 카드 80개 발견
📍  1위 [Up] VTcosmetic楽天市場店 | ＼最大49％OFF＋ギフト＋送料無料／総合ランキング1位【V | ⭐4.5 (7886건)
📍  2위 [Down] アテニア公式ショップ　楽天市場 | 【ポイント10倍！〜3月11日1:59】スキンクリア クレン | ⭐4.5 (44082건)
📍  3위 [Up] VTcosmetic楽天市場店 | 【クーポン付き】＼最大68％OFF＋現品ギフト＋送料無料／【 | ⭐4.5 (1520건)

🚀 [【クーポン付き】＼最大68％OFF＋現品ギフト＋送料無料／【VT公式】【楽天限定】【春肌 開花宣言! 華やかフルケアセッ…] 리뷰 전체 수집 시작... (shop=371041, item=10001951)
🔄 53페이지 완료... (누적 1570개)
✨ 리뷰 수집 완료: 총 1570개
📍  4위 [Up] 【公式】Yunth Store | 【P30%還元+セット9日23:59マデ】【公式】Yunth | ⭐4.5 (43354건)
📍  5위 [Up] ANUA Official 楽 | ★58％OFF+現品ギフト+先着ギフトまで★【ROOMコラボ | ⭐3 (2건)
📍  6위 [Up] コスメティック　やよい | ＼超トクも残り2日!P20%還元確定+10%OFF／【資生堂 | ⭐4.5 (1778건)
📍  7위 [Down] アテニア公式ショップ　楽天市場 | 【ポイント5倍！〜3月11日1:59】スキンクリア クレンズ | ⭐4.5 (37941건)
📍  8위 [Up] VTcosmetic楽天市場店 | ＼最大58％OFF＋ギフト＋送料無料／【 VT リードルショ | ⭐4.5 (506건)
📍  9위 [Stay] シュウ ウエムラ 公式ショップ | 【ポイント10倍｜3/4 20:00-3/11 1:59】総 | ⭐4.5 (31569건)
📍 10위 [Up] タカミ 公式ショップ楽天市場店 | 【ポイント10倍│楽天スーパーSALE限定販売】タカミ角質美 | ⭐4.5 (433건)

🏆 라쿠텐 스킨케어 랭킹 Top 10 (주간)
 1위 [Up] | VTcosmetic楽天市場店 | ＼最大

,Page,Nickname,Rating,Body,PostDate,Age,Sex,Sku
0,1,rumirumi7さん,5,スーパーセールの時にお得に購入できて嬉しいです。\nセールのタイミングだと欲しかった化粧水と...,2025/09/13,40代,female,セット 選択:リッチな大人のツヤ肌セット | ボックス選択:なし
1,1,購入者さん,2,母に頼まれて即効！ハリ弾力肌のセットを購入しました。\nセール中だったのでかなりお得ですが、...,2025/04/11,,None,セット 選択:即光！ハリ弾力肌セット 2025 | ボックス選択:なし
2,1,oui1515さん,5,届くのは思ったより全然早かったです！PDRN使ってみたかったのでお得に買えてよかったです。\...,2025/07/06,40代,female,セット 選択:リッチな大人のツヤ肌セット | ボックス選択:限定ボックスあり
3,1,のびちゃん919さん,5,安いのでサンプルサイズかと思ってましたが、\n普通に来たのでびつくり！保湿も抜群！\nこのコ...,2025/08/11,50代,female,セット 選択:リッチな大人のツヤ肌セット | ボックス選択:限定ボックスあり
4,1,にゃん太xxxさん,5,去年からスーパーセールの度に限定セットを購入してます。。\n限定セットがお得すぎて、3か月に...,2025/09/27,30代,female,セット 選択:ゆらぎケアセット | ボックス選択:限定ボックスあり
